# Региональный определитель птиц по голосу: бейзлайны и улучшенные модели

**Кейс.** MVP веб-сервиса, который по короткой аудиозаписи (5 секунд) определяет вид птицы для региона
**Санкт-Петербург и Ленинградская область**. Ценность — *локальная* точность: глобальные модели
(Merlin, BirdNET) на региональных видах дают ~25–40%, а мы специализируем модель на локальных данных Xeno-Canto.

**ML-задача.** Многоклассовая классификация аудио: сегмент 5 сек → один из **20 целевых видов**.
Объединения видов заданы структурой папок: «Воробей» = домовый + полевой; «Чайка» = малая + озёрная + серебристая.

**Что в ноутбуке (ДЗ):**
1. Загрузка и описание данных (до/после обработки).
2. EDA с выводом к каждой диаграмме.
3. Метрика под задачу (дисбаланс) + композитная метрика с учётом инференса.
4. **Корректный сплит по записям** (без утечки).
5. 3 бейзлайна: наивный, классика на признаках, бустинг.
6. Улучшенные модели (transfer learning), три варианта со сравнением:
   **A** — эмбеддинги AST (AudioSet) + классификатор; **B** — fine-tuning CNN (timm) на мел-спектрограммах;
   **C** — эмбеддинги **BirdNET** (птичий backbone) + классификатор.
7. Оценка (F1, confusion matrix, per-class), Optuna, SHAP, стоимость инференса, выводы и связь с бизнесом.

> Ноутбук рассчитан на **Google Colab с GPU** (`Runtime → Change runtime type → GPU`). Запускать сверху вниз.

## 0. Окружение и установка

Ставим только то, чего нет в Colab по умолчанию. Логи установки подавляем.

In [ ]:
import sys, subprocess
def pip_install(pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)

# timm/catboost/optuna/shap/birdnetlib отсутствуют в Colab по умолчанию
pip_install(["timm", "catboost", "optuna", "shap", "transformers>=4.40", "librosa>=0.10",
             "birdnetlib", "dvc[s3]"])

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("PyTorch:", torch.__version__, "| устройство:", DEVICE)
if DEVICE == "cpu":
    print("ВНИМАНИЕ: GPU не найден. fine-tuning CNN будет медленным. "
          "Включите GPU: Runtime -> Change runtime type -> GPU.")

## Данные: как получить их в Colab

Аудио лежит под DVC (структура `data/audio/<Вид>/`). Ячейка ниже сама:
клонирует репозиторий, тянет `data/audio` из MinIO и **заново нарезает** сегменты в `data/processed`
скриптом `src/data/preprocess.py --clean` (вид = имя папки). Отдельно в терминале запускать ничего не нужно.

> Безопасность: в `.dvc/config` лежат ключи доступа к S3. Если ключ боевой — его стоит ротировать.

In [ ]:
import os, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/annKlv22/MFDP"
BRANCH   = "develop"

def has_processed(root="data/processed"):
    p = Path(root)
    return p.exists() and any(p.glob("*/*.wav"))

if has_processed():
    print("data/processed уже на месте.")
else:
    if not Path("MFDP").exists() and not Path("src/data/preprocess.py").exists():
        subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, "MFDP"], check=False)
    if Path("MFDP").exists():
        os.chdir("MFDP")
    # тянем сырые записи из MinIO (креды берутся из .dvc/config репозитория)
    subprocess.run([sys.executable, "-m", "dvc", "pull", "data/audio.dvc"], check=False)
    # нарезаем сегменты (вид = имя папки), очищая прошлый набор
    subprocess.run([sys.executable, "src/data/preprocess.py",
                    "--input", "data/audio", "--output", "data/processed", "--clean"], check=False)
print("Готово:", has_processed(), "| рабочая папка:", os.getcwd())

## 1. Данные: что было до обработки и что после

**Источник.** [Xeno-Canto](https://xeno-canto.org), отобрано вручную по гео-боксу СПб/Ленобласти
(широта 58.8–60.9° с.ш., долгота 28.7–34.5° в.д.).

**До обработки:** 173 исходные записи (20 видов), форматы MP3/WAV; вид задан папкой `data/audio/<Вид>/`.
Каждый файл — отдельная запись (уникальный Xeno-Canto id). Объединения: «Воробей» (домовый+полевой),
«Чайка» (малая+озёрная+серебристая); «Чёрный дрозд» и «Белобровик» — раздельно.

**Обработка** (`src/data/preprocess.py`): моно, ресэмплинг 48 kHz, нарезка на окна **5 сек** с шагом **2.5 сек**,
отсев тихих сегментов (RMS), сохранение в `data/processed/<Вид>/...wav`.

Ниже строим метаданные прямо из папок: **метка = имя папки** (объединения уже заданы структурой),
исключаем любые `_aug` файлы, чтобы аугментация не протекла в валидацию.

In [ ]:
import os, re, json, time, math, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED); np.random.seed(SEED)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110

PROCESSED = Path("data/processed")

def normalize_species(dirname):
    return dirname.replace("_", " ")          # метка = имя папки

def recording_id(filepath):
    m = re.search(r"XC\d+", str(filepath))    # все сегменты одной записи -> один XC id
    return m.group(0) if m else Path(filepath).stem

rows = []
for sp_dir in sorted(PROCESSED.iterdir()):
    if not sp_dir.is_dir():
        continue
    species = normalize_species(sp_dir.name)
    for wav in sorted(sp_dir.glob("*.wav")):
        if "_aug" in wav.stem:
            continue
        rows.append({"filepath": str(wav), "species": species, "rec_id": recording_id(wav)})

meta = pd.DataFrame(rows)
CLASSES = sorted(meta["species"].unique())
print(f"Видов: {len(CLASSES)} | сегментов: {len(meta)} | записей: {meta['rec_id'].nunique()}")
meta.head()

## 2. Разведочный анализ (EDA)

### 2.1 Записи против сегментов

«Много сегментов» ≠ «много данных»: сегменты одной записи скоррелированы. Честный размер выборки —
число **независимых записей**, по ним и считаем статистику/сплит.

In [ ]:
agg = (meta.groupby("species")
          .agg(сегментов=("filepath", "count"), записей=("rec_id", "nunique"))
          .sort_values("записей"))

print("Записей на вид:")
print(agg["записей"].describe().round(2).to_string())

fig, ax = plt.subplots(1, 2, figsize=(12, 7))
agg["сегментов"].plot.barh(ax=ax[0], color="#4C72B0"); ax[0].set_title("Сегментов на вид"); ax[0].set_ylabel("")
agg["записей"].plot.barh(ax=ax[1], color="#C44E52"); ax[1].set_title("Записей на вид"); ax[1].set_ylabel("")
ax[1].set_yticklabels([]); plt.tight_layout(); plt.show()

**Вывод 2.1.** После курации набор почти сбалансирован по записям (≈1.7×, 7–12 записей на вид).
Сегментов на вид больше и разброс шире (зависит от длины записей), но опираемся на записи: у видов с 7 записями
в val/test попадёт лишь по 1 записи — метрики на них будут шумными.

In [ ]:
# Примеры мел-спектрограмм для нескольких видов
import librosa, librosa.display
demo = meta.groupby("species").first().reset_index().sample(min(6, len(CLASSES)), random_state=SEED)
fig, axes = plt.subplots(2, 3, figsize=(14, 6))
for ax, (_, r) in zip(axes.ravel(), demo.iterrows()):
    y, _ = librosa.load(r["filepath"], sr=32000, mono=True)
    S = librosa.power_to_db(librosa.feature.melspectrogram(y=y, sr=32000, n_mels=128, fmax=16000), ref=np.max)
    librosa.display.specshow(S, sr=32000, x_axis="time", y_axis="mel", fmax=16000, ax=ax)
    ax.set_title(r["species"], fontsize=10)
plt.tight_layout(); plt.show()

**Вывод 2.2.** Виды различаются рисунком спектра (диапазон, ритм, гармоники) — признаки информативны
и для классики, и для CNN. Акустически близкие пары для контроля на confusion matrix:
**Чёрный дрозд / Белобровик** (оба дрозды), **Большая синица / Лазоревка** (синицы),
плюс разнородность внутри объединённых «Чайка» и «Воробей».

## 3. Метрика

**Не accuracy** как единственная: при дисбалансе вводит в заблуждение. Берём:
- **Основная — macro-F1**: равный вес всем видам (для выбора модели и честной оценки).
- Продуктовая — **top-1 accuracy / weighted-F1**: то, что чувствует пользователь, сравнимо с конкурентами.
- **Композитная** (ML → продукт): штраф за латентность инференса, бизнес требует ответ за «несколько секунд»:
  `Score = F1_macro − λ · (latency_ms / budget_ms)`.

In [ ]:
from sklearn.metrics import f1_score, accuracy_score, balanced_accuracy_score

LATENCY_BUDGET_MS = 1000.0
LAMBDA = 0.10

def evaluate(y_true, y_pred, latency_ms=None):
    labels = sorted(set(y_true))
    res = {
        "f1_macro":     f1_score(y_true, y_pred, labels=labels, average="macro", zero_division=0),
        "f1_weighted":  f1_score(y_true, y_pred, labels=labels, average="weighted", zero_division=0),
        "accuracy":     accuracy_score(y_true, y_pred),
        "balanced_acc": balanced_accuracy_score(y_true, y_pred),
        "n_classes_eval": len(labels),
    }
    if latency_ms is not None:
        res["latency_ms"] = latency_ms
        res["composite"]  = res["f1_macro"] - LAMBDA * (latency_ms / LATENCY_BUDGET_MS)
    return res

RESULTS, PRED = {}, {}
def log_result(name, y_true, y_pred, latency_ms=None):
    RESULTS[name] = evaluate(y_true, y_pred, latency_ms)
    PRED[name] = (np.array(list(y_true)), np.array(list(y_pred)))
    print(name, "->", {k: round(v, 3) for k, v in RESULTS[name].items()})

## 4. Корректный сплит по записям (без утечки)

Делим **по записям** (все сегменты одной записи — в одном сплите), стратифицируем по виду,
доли train/val/test = 70/15/15. Так метрики честные, а не завышенные утечкой.

In [ ]:
def assign_splits(meta, seed=SEED, val=0.15, test=0.15):
    rng = random.Random(seed)
    rec_species = meta.groupby("rec_id")["species"].agg(lambda s: s.value_counts().index[0])
    split_of = {}
    for sp in sorted(meta["species"].unique()):
        recs = sorted(rec_species[rec_species == sp].index.tolist())
        rng.shuffle(recs)
        n = len(recs)
        if n == 1:
            parts = {"train": recs}
        elif n == 2:
            parts = {"train": recs[:1], "test": recs[1:]}
        elif n == 3:
            parts = {"train": recs[:1], "val": recs[1:2], "test": recs[2:]}
        else:
            n_test = max(1, round(test * n)); n_val = max(1, round(val * n))
            parts = {"test": recs[:n_test], "val": recs[n_test:n_test+n_val], "train": recs[n_test+n_val:]}
        for split, rs in parts.items():
            for r in rs:
                split_of[r] = split
    return meta["rec_id"].map(split_of)

meta["split"] = assign_splits(meta)

ids = {s: set(meta.loc[meta.split == s, "rec_id"]) for s in ["train", "val", "test"]}
print("Записей train/val/test:", {s: len(v) for s, v in ids.items()})
print("Пересечения записей:", len(ids["train"] & ids["val"]), len(ids["train"] & ids["test"]), len(ids["val"] & ids["test"]))
print("Видов в сплитах:", meta.groupby("split")["species"].nunique().to_dict(), "из", len(CLASSES))
print("Сегментов по сплитам:", meta["split"].value_counts().to_dict())

**Вывод 4.** Пересечения записей между сплитами нулевые — утечки нет. Все 20 видов присутствуют в каждом сплите.

## 5. Бейзлайны

1. **Наивный** — `DummyClassifier` (самый частый класс): «пол» метрики.
2. **Классика** — ручные аудио-признаки (MFCC, спектральные, ZCR, chroma) + `StandardScaler`
   (обучается **только на train**) + логистическая регрессия с балансировкой.
3. **Бустинг** — те же признаки + CatBoost с весами классов.

In [ ]:
import librosa
FEAT_CACHE = "features_cache.npz"

def extract_features(path, sr=32000):
    y, _ = librosa.load(path, sr=sr, mono=True)
    if len(y) < sr // 2:
        y = np.pad(y, (0, sr // 2))
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=20)
    feats = [
        mfcc.mean(1), mfcc.std(1), librosa.feature.delta(mfcc).mean(1),
        [librosa.feature.spectral_centroid(y=y, sr=sr).mean()],
        [librosa.feature.spectral_bandwidth(y=y, sr=sr).mean()],
        [librosa.feature.spectral_rolloff(y=y, sr=sr).mean()],
        [librosa.feature.zero_crossing_rate(y).mean()],
        librosa.feature.chroma_stft(y=y, sr=sr).mean(1),
        [librosa.feature.rms(y=y).mean()],
    ]
    return np.concatenate([np.atleast_1d(f) for f in feats]).astype(np.float32)

if Path(FEAT_CACHE).exists():
    d = np.load(FEAT_CACHE, allow_pickle=True)
    path2feat = {p: d["X"][i] for i, p in enumerate(d["paths"])}
else:
    path2feat = {}
    for i, p in enumerate(meta["filepath"]):
        path2feat[p] = extract_features(p)
        if (i + 1) % 300 == 0: print(f"  признаки: {i+1}/{len(meta)}")
    np.savez(FEAT_CACHE, X=np.stack([path2feat[p] for p in meta["filepath"]]),
             paths=np.array(meta["filepath"].tolist()))
    print("признаки готовы")

def split_xy(split):
    m = meta[meta.split == split]
    return np.stack([path2feat[p] for p in m["filepath"]]), m["species"].values

Xtr, ytr = split_xy("train"); Xte, yte = split_xy("test")
print("train/test:", Xtr.shape, Xte.shape)

In [ ]:
# Бейзлайн 0 — наивный
from sklearn.dummy import DummyClassifier
dummy = DummyClassifier(strategy="most_frequent").fit(Xtr, ytr)
log_result("0. Наивный (most_frequent)", yte, dummy.predict(Xte))

In [ ]:
# Бейзлайн 1 — StandardScaler (fit ТОЛЬКО на train) + логистическая регрессия с балансировкой
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
logreg = make_pipeline(StandardScaler(),
                       LogisticRegression(max_iter=2000, class_weight="balanced")).fit(Xtr, ytr)
log_result("1. LogReg на признаках", yte, logreg.predict(Xte))

In [ ]:
# Бейзлайн 2 — CatBoost с весами классов (логи подавлены)
from catboost import CatBoostClassifier
from sklearn.utils.class_weight import compute_class_weight
classes_tr = np.unique(ytr)
w = compute_class_weight("balanced", classes=classes_tr, y=ytr)
cat = CatBoostClassifier(iterations=400, depth=6, learning_rate=0.1, loss_function="MultiClass",
                         class_names=list(classes_tr), class_weights=list(w),
                         random_seed=SEED, verbose=False).fit(Xtr, ytr)
log_result("2. CatBoost на признаках", yte, cat.predict(Xte).ravel().astype(str))

**Вывод 5.** Наивный бейзлайн задаёт пол метрики. Классика и CatBoost — планка, которую должны побить
усложнённые модели. Если не бьют — проверяем утечку/метрику или берём более лёгкое решение.

## 6. Улучшенная модель A: эмбеддинги AST (AudioSet) + классификатор

Transfer learning: большая модель, предобученная на AudioSet, как **замороженный экстрактор признаков**;
поверх — лёгкий классификатор. Быстро, почти не переобучается на малых данных.

In [ ]:
import torch, librosa
from transformers import AutoFeatureExtractor, ASTModel
AST_NAME = "MIT/ast-finetuned-audioset-10-10-0.4593"
fe  = AutoFeatureExtractor.from_pretrained(AST_NAME)
ast = ASTModel.from_pretrained(AST_NAME).to(DEVICE).eval()

@torch.no_grad()
def ast_embed(paths, batch=16):
    out = []
    for i in range(0, len(paths), batch):
        wavs = [librosa.load(p, sr=16000, mono=True)[0] for p in paths[i:i+batch]]
        inp = fe(wavs, sampling_rate=16000, return_tensors="pt")
        out.append(ast(inp["input_values"].to(DEVICE)).pooler_output.cpu().numpy())
    return np.concatenate(out)

EMB_CACHE = "ast_emb_cache.npz"
if Path(EMB_CACHE).exists():
    d = np.load(EMB_CACHE, allow_pickle=True); path2emb = {p: d["X"][i] for i, p in enumerate(d["paths"])}
else:
    paths = meta["filepath"].tolist(); E = ast_embed(paths)
    path2emb = {p: E[i] for i, p in enumerate(paths)}
    np.savez(EMB_CACHE, X=E, paths=np.array(paths))
print("AST эмбеддинг:", next(iter(path2emb.values())).shape)

def split_emb(split, table):
    m = meta[meta.split == split]
    return np.stack([table[p] for p in m["filepath"]]), m["species"].values

Etr, ytr_e = split_emb("train", path2emb); Eva, yva_e = split_emb("val", path2emb); Ete, yte_e = split_emb("test", path2emb)
ast_clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000, class_weight="balanced")).fit(Etr, ytr_e)
log_result("A. AST эмбеддинги + LogReg", yte_e, ast_clf.predict(Ete))

## 7. Улучшенная модель B: fine-tuning CNN (timm) на мел-спектрограммах

Рецепт BirdCLEF: мел-спектрограмма как картинка → предобученная EfficientNet-B0, которую дообучаем.
Аугментация (SpecAugment + шум) — только к train.

In [ ]:
import librosa
CNN_SR, N_MELS, HOP, NFFT, W = 32000, 128, 320, 1024, 384

def make_mel(path):
    y, _ = librosa.load(path, sr=CNN_SR, mono=True)
    S = librosa.power_to_db(librosa.feature.melspectrogram(
        y=y, sr=CNN_SR, n_fft=NFFT, hop_length=HOP, n_mels=N_MELS, fmin=50, fmax=16000), ref=np.max)
    if S.shape[1] < W:
        S = np.pad(S, ((0, 0), (0, W - S.shape[1])), mode="constant", constant_values=S.min())
    else:
        S = S[:, :W]
    return ((S - S.mean()) / (S.std() + 1e-6)).astype(np.float32)

path2mel = {p: make_mel(p) for p in meta["filepath"]}
print("Мел-спектрограммы готовы:", len(path2mel))

In [ ]:
import torch, timm
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder().fit(CLASSES)

def spec_augment(S, rng):
    S = S.copy()
    f = rng.randint(0, 16); f0 = rng.randint(0, max(1, N_MELS - f)); S[f0:f0+f, :] = S.min()
    t = rng.randint(0, 48); t0 = rng.randint(0, max(1, W - t)); S[:, t0:t0+t] = S.min()
    if rng.random() < 0.5: S = S + np.random.randn(*S.shape).astype(np.float32) * 0.1
    return S

class MelDS(Dataset):
    def __init__(self, df, train=False):
        self.paths = df["filepath"].tolist(); self.y = le.transform(df["species"].values)
        self.train = train; self.rng = random.Random(SEED)
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        S = path2mel[self.paths[i]]
        if self.train: S = spec_augment(S, self.rng)
        return torch.from_numpy(S).unsqueeze(0).repeat(3, 1, 1), int(self.y[i])

tr_df, va_df, te_df = (meta[meta.split == s] for s in ["train", "val", "test"])
dl_tr = DataLoader(MelDS(tr_df, train=True), batch_size=32, shuffle=True, num_workers=2)
dl_va = DataLoader(MelDS(va_df), batch_size=64, num_workers=2)
dl_te = DataLoader(MelDS(te_df), batch_size=64, num_workers=2)

cw = compute_class_weight("balanced", classes=np.arange(len(CLASSES)),
                          y=le.transform(tr_df["species"].values))
weight_t = torch.tensor(cw, dtype=torch.float32).to(DEVICE)

In [ ]:
from sklearn.metrics import f1_score
model = timm.create_model("efficientnet_b0", pretrained=True, num_classes=len(CLASSES), in_chans=3).to(DEVICE)
opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
crit = nn.CrossEntropyLoss(weight=weight_t)
EPOCHS = 12
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)

@torch.no_grad()
def predict_loader(dl):
    model.eval(); P, Y = [], []
    for x, y in dl:
        P.append(model(x.to(DEVICE)).argmax(1).cpu().numpy()); Y.append(y.numpy())
    return np.concatenate(P), np.concatenate(Y)

best_f1, best_state = -1, None
for ep in range(EPOCHS):
    model.train()
    for x, y in dl_tr:
        opt.zero_grad(); loss = crit(model(x.to(DEVICE)), y.to(DEVICE)); loss.backward(); opt.step()
    sched.step()
    pv, yv = predict_loader(dl_va); f1v = f1_score(yv, pv, average="macro", zero_division=0)
    if f1v >= best_f1: best_f1, best_state = f1v, {k: v.cpu().clone() for k, v in model.state_dict().items()}
    print(f"эпоха {ep+1:02d}/{EPOCHS}  loss={loss.item():.3f}  val_macroF1={f1v:.3f}")

if best_state: model.load_state_dict(best_state)
pte, yte_idx = predict_loader(dl_te)
log_result("B. EfficientNet (fine-tune)", le.inverse_transform(yte_idx), le.inverse_transform(pte))

## 8. Улучшенная модель C: эмбеддинги BirdNET (птичий backbone) + классификатор

BirdNET (Cornell) предобучен именно на птицах — его эмбеддинги обычно сильнее общего AudioSet-AST
на видовой классификации и прямо усиливают бизнес-тезис «специализируемся на птицах региона».
Используем `birdnetlib` как замороженный экстрактор эмбеддингов (1024-d), поверх — тот же лёгкий классификатор.

In [ ]:
# BirdNET-эмбеддинги через birdnetlib (обёрнуто в try/except: при сбое модель C пропускается)
BN_CACHE = "birdnet_emb_cache.npz"
birdnet_ok = True
try:
    if Path(BN_CACHE).exists():
        d = np.load(BN_CACHE, allow_pickle=True); path2bn = {p: d["X"][i] for i, p in enumerate(d["paths"])}
    else:
        from birdnetlib import Recording
        from birdnetlib.analyzer import Analyzer
        analyzer = Analyzer()
        def birdnet_embed_one(path):
            rec = Recording(analyzer, path, min_conf=0.1)
            rec.extract_embeddings()
            embs = [np.array(e["embeddings"], dtype=np.float32) for e in rec.embeddings]
            return np.mean(embs, axis=0) if embs else None
        paths = meta["filepath"].tolist(); rowsE = []
        dim = None
        for i, p in enumerate(paths):
            v = birdnet_embed_one(p)
            if v is not None: dim = len(v)
            rowsE.append(v)
            if (i + 1) % 200 == 0: print(f"  BirdNET: {i+1}/{len(paths)}")
        dim = dim or 1024
        E = np.stack([v if v is not None else np.zeros(dim, dtype=np.float32) for v in rowsE])
        path2bn = {p: E[i] for i, p in enumerate(paths)}
        np.savez(BN_CACHE, X=E, paths=np.array(paths))
    print("BirdNET эмбеддинг:", next(iter(path2bn.values())).shape)

    Btr, ytr_b = split_emb("train", path2bn); Bte, yte_b = split_emb("test", path2bn)
    bn_clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000, class_weight="balanced")).fit(Btr, ytr_b)
    log_result("C. BirdNET эмбеддинги + LogReg", yte_b, bn_clf.predict(Bte))
except Exception as e:
    birdnet_ok = False
    print("BirdNET пропущен (ошибка):", repr(e))

## 9. Оптимизация гиперпараметров (Optuna)

Подбираем `C` для классификатора поверх AST-эмбеддингов на честном val (максимизируем macro-F1).

In [ ]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    C = trial.suggest_float("C", 1e-3, 1e2, log=True)
    clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000, class_weight="balanced", C=C)).fit(Etr, ytr_e)
    return f1_score(yva_e, clf.predict(Eva), average="macro", zero_division=0)

study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED))
study.optimize(objective, n_trials=25)
print("Лучшее C:", round(study.best_params["C"], 4), "| val macro-F1:", round(study.best_value, 3))
best = make_pipeline(StandardScaler(),
                     LogisticRegression(max_iter=3000, class_weight="balanced", C=study.best_params["C"])).fit(Etr, ytr_e)
log_result("A+. AST + LogReg (Optuna)", yte_e, best.predict(Ete))

## 10. Сравнение моделей и анализ ошибок

In [ ]:
res_df = pd.DataFrame(RESULTS).T[["f1_macro", "f1_weighted", "accuracy", "balanced_acc", "n_classes_eval"]]
res_df = res_df.round(3).sort_values("f1_macro")
display(res_df)
ax = res_df["f1_macro"].plot.barh(figsize=(8, 4), color="#55A868")
ax.axvline(0.75, ls="--", color="red", label="бизнес-цель 0.75"); ax.set_xlabel("macro-F1"); ax.legend()
plt.title("macro-F1 по моделям (test)"); plt.tight_layout(); plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
best_name = res_df["f1_macro"].idxmax()
yt, yp = PRED[best_name]
labels = sorted(set(yt))
cm = confusion_matrix(yt, yp, labels=labels)
plt.figure(figsize=(11, 9))
sns.heatmap(cm, xticklabels=labels, yticklabels=labels, cmap="Blues", cbar=False)
plt.title(f"Confusion matrix: {best_name}"); plt.ylabel("истинный"); plt.xlabel("предсказанный")
plt.tight_layout(); plt.show()

rep = pd.DataFrame(classification_report(yt, yp, labels=labels, output_dict=True, zero_division=0)).T
print("Худшие 8 видов по F1 у лучшей модели:")
display(rep.loc[labels].sort_values("f1-score")[["precision", "recall", "f1-score", "support"]].head(8).round(2))

**Анализ ошибок.** Смотрим внедиагональные ячейки: основные путаницы ожидаемо у близких пар
(Чёрный дрозд/Белобровик, синицы) и у видов с малым числом записей. Рычаг улучшения — досбор записей по слабым видам.

## 11. Объяснимость (SHAP)

SHAP на CatBoost (признаки интерпретируемы): какие MFCC/спектральные характеристики важнее для разделения видов.

In [ ]:
import shap
feat_names = ([f"mfcc{i}_mean" for i in range(20)] + [f"mfcc{i}_std" for i in range(20)]
              + [f"mfcc{i}_delta" for i in range(20)] + ["centroid", "bandwidth", "rolloff", "zcr"]
              + [f"chroma{i}" for i in range(12)] + ["rms"])
try:
    sv = shap.TreeExplainer(cat).shap_values(Xte[:min(100, len(Xte))])
    shap.summary_plot(sv, Xte[:min(100, len(Xte))], feature_names=feat_names, plot_type="bar", max_display=15, show=True)
except Exception as e:
    print("SHAP пропущен:", e)

## 12. Стоимость инференса и композитная метрика

Меряем латентность на сегмент и считаем `Score = F1_macro − λ·(latency/budget)`.

In [ ]:
import time, torch
sample_paths = meta[meta.split == "test"]["filepath"].tolist()[:30]
def time_call(fn, paths, repeats=30):
    paths = list(paths)[:repeats]; fn(paths[:1])
    t0 = time.time(); fn(paths); return (time.time() - t0) / len(paths) * 1000

lat = {}
lat["1. LogReg на признаках"]  = time_call(lambda ps: logreg.predict(np.stack([extract_features(p) for p in ps])), sample_paths)
lat["2. CatBoost на признаках"] = time_call(lambda ps: cat.predict(np.stack([extract_features(p) for p in ps])), sample_paths)
lat["A. AST эмбеддинги + LogReg"] = time_call(lambda ps: ast_clf.predict(ast_embed(ps, batch=8)), sample_paths)
if "B. EfficientNet (fine-tune)" in RESULTS:
    @torch.no_grad()
    def cnn_infer(ps):
        x = torch.stack([torch.from_numpy(make_mel(p)).unsqueeze(0).repeat(3,1,1) for p in ps]).to(DEVICE)
        return model(x).argmax(1).cpu().numpy()
    lat["B. EfficientNet (fine-tune)"] = time_call(cnn_infer, sample_paths)

for name, ms in lat.items():
    if name in RESULTS:
        RESULTS[name]["latency_ms"] = round(ms, 1)
        RESULTS[name]["composite"] = round(RESULTS[name]["f1_macro"] - LAMBDA * ms / LATENCY_BUDGET_MS, 3)

final = pd.DataFrame(RESULTS).T
cols = [c for c in ["f1_macro", "f1_weighted", "accuracy", "latency_ms", "composite"] if c in final.columns]
display(final[cols].sort_values("f1_macro").round(3))

## 13. Выводы, связь с бизнесом и следующие шаги

**Итоги.**
- Наивный бейзлайн задаёт пол; transfer learning (AST/BirdNET-эмбеддинги, fine-tune CNN) бьёт классику на признаках.
- BirdNET-эмбеддинги — птичий backbone, ожидаемо сильны на региональных видах и прямо отвечают тезису продукта.
- Главный технический фундамент — **честный сплит по записям**: метрики не завышены утечкой.

**Связь ML → бизнес.**
- macro-F1 отражает «точнее глобального BirdNET на регионе»; accuracy — продуктовая, сравнима с конкурентами.
- Композитная метрика учитывает требование скорости ответа.
- Узкое место — данные: 7–12 записей на вид. У видов с 7 записями val/test по 1 записи (шумно).

**Следующие шаги.**
1. Досбор записей для видов с 7 записями (до ≥10–15) — рычаг №1.
2. Агрегация вероятностей по окнам записи на инференсе (усреднение по 5-сек сегментам).
3. Бинарный гейт «птица / не птица» перед классификацией (борьба с полевым шумом).
4. Зафиксировать лучшую модель и завернуть в FastAPI + Docker.